# PlantWild Scratch Models: Final Holdout

This notebook evaluates the five frozen random-initialization, three-seed scratch comparisons exactly once. It starts in preflight mode and does not run final-holdout inference until explicitly unlocked.

In [1]:
# Use preflight first. Sealed evaluation is a one-time final comparison.
RUN_MODE = 'sealed_evaluation'  # preflight | sealed_evaluation | report_existing
ACKNOWLEDGEMENT = 'RUN_PLANTWILD_SCRATCH_FINAL_HOLDOUT_ONCE'
SEALED_ACKNOWLEDGEMENT = 'RUN_PLANTWILD_SCRATCH_FINAL_HOLDOUT_ONCE'

DEVICE = 'cpu'
BATCH_SIZE = 64
NUM_WORKERS = 0


In [2]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'code' / 'research_evaluation' / 'sealed_scratch.py').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook inside the Agrivision4 workspace')

REPO_ROOT = find_repo_root(Path.cwd())
CODE_ROOT = REPO_ROOT / "code"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
print({'repo_root': str(REPO_ROOT), 'run_mode': RUN_MODE, 'device': DEVICE})


{'repo_root': '/run/media/sha-weed/adc6c87e-0116-4a25-a6e9-8555193c1e51/Research/Agrivision4', 'run_mode': 'sealed_evaluation', 'device': 'cpu'}


## Execute

In final mode, the evaluator verifies the frozen manifest/checkpoint contract, then writes all results atomically. Progress is printed after each of the 15 checkpoint evaluations.

In [3]:
from research_evaluation.sealed_scratch import build_spec, execute

spec = build_spec(REPO_ROOT, 'plantwild')
result = execute(
    spec,
    run_mode=RUN_MODE,
    acknowledgement=ACKNOWLEDGEMENT,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print({'status': result['status'], 'sealed_inference': result.get('sealed_inference_executed', False)})


/home/sha-weed/.local/share/mamba/envs/.venv_ensemble/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{"event": "checkpoint_evaluated", "index": 1, "total": 15, "model_key": "wavestarnet_big_transfer_v2", "seed": 42, "accuracy": 0.5095206828627709, "macro_f1": 0.4428564115206916}
{"event": "checkpoint_evaluated", "index": 2, "total": 15, "model_key": "wavestarnet_big_transfer_v2", "seed": 2024, "accuracy": 0.5180564674983585, "macro_f1": 0.45552886714185176}
{"event": "checkpoint_evaluated", "index": 3, "total": 15, "model_key": "wavestarnet_big_transfer_v2", "seed": 3407, "accuracy": 0.5285620485883126, "macro_f1": 0.4564998881223476}
{"event": "checkpoint_evaluated", "index": 4, "total": 15, "model_key": "efficientnet_b0", "seed": 42, "accuracy": 0.39461588969139855, "macro_f1": 0.33064976566754867}
{"event": "checkpoint_evaluated", "index": 5, "total": 15, "model_key": "efficientnet_b0", "seed": 2024, "accuracy": 0.4248194353250164, "macro_f1": 0.3573342803744962}
{"event": "checkpoint_evaluated", "index": 6, "total": 15, "model_key": "efficientnet_b0", "seed": 3407, "accuracy": 0.4

## Principal Results

The tables below remain embedded after you save the executed notebook.

In [4]:
import pandas as pd
from IPython.display import Markdown, display

if RUN_MODE == 'preflight':
    protocol = result['protocol']
    display(Markdown('### Preflight passed: no sealed inference executed'))
    display(pd.DataFrame([{
        'final_holdout_samples': protocol['test_samples'],
        'classes': protocol['class_count'],
        'minimum_class_support': protocol['minimum_class_support'],
        'checkpoint_count': result['checkpoint_count'],
        'strict_loads': len(result['strict_load_audits']),
    }]))
    display(pd.DataFrame(result['checkpoint_table'])[['display_name', 'model_key', 'seed', 'parameters', 'checkpoint_sha256']])
else:
    display(Markdown('### Final aggregate ranking: mean, SD, and 95% t CI'))
    display(result['aggregate'])
    display(Markdown('### Per-seed final-holdout metrics'))
    display(result['seed_metrics'])
    if 'artifact_inventory' in result:
        display(Markdown(f"### Output inventory: {len(result['artifact_inventory'])} files"))
        display(pd.DataFrame(result['artifact_inventory']))
    display(Markdown(f"**Saved to:** `{result['output_directory']}`"))


### Final aggregate ranking: mean, SD, and 95% t CI

,rank_by_macro_f1,model_key,display_name,parameters,seeds,seed_count,accuracy_mean,accuracy_std,accuracy_ci95_low,accuracy_ci95_high,...,nll_ci95_low,nll_ci95_high,brier_mean,brier_std,brier_ci95_low,brier_ci95_high,ece_15bin_mean,ece_15bin_std,ece_15bin_ci95_low,ece_15bin_ci95_high
0,1,mobilevit_s,MobileViT-S,5004937,"42,2024,3407",3,0.525498,0.008817,0.503594,0.547401,...,1.900184,2.046196,0.637023,0.004000,0.627086,0.646961,0.096247,0.004534,0.084983,0.107511
1,2,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,3013109,"42,2024,3407",3,0.518713,0.009538,0.495020,0.542406,...,2.009199,2.222072,0.648055,0.009433,0.624623,0.671488,0.096664,0.001335,0.093347,0.099981
2,3,efficientnet_b3,EfficientNet-B3,10857617,"42,2024,3407",3,0.448457,0.003283,0.440302,0.456612,...,2.400611,2.613742,0.712780,0.003361,0.704431,0.721130,0.067605,0.007728,0.048407,0.086803
3,4,efficientnet_b0,EfficientNet-B0,4142053,"42,2024,3407",3,0.411906,0.015570,0.373228,0.450585,...,2.565542,2.900627,0.764590,0.021003,0.712415,0.816766,0.095330,0.015607,0.056560,0.134100
4,5,mobilenetv3_large_100,MobileNetV3-Large,4336537,"42,2024,3407",3,0.395710,0.005922,0.381000,0.410420,...,2.737789,2.909595,0.778711,0.000714,0.776937,0.780486,0.124031,0.005211,0.111087,0.136975


### Per-seed final-holdout metrics

,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,weighted_precision,weighted_recall,nll,brier,ece_15bin,model_key,display_name,seed,parameters,runtime_seconds,checkpoint_sha256
0,0.509521,0.452545,0.442856,0.505218,0.456501,0.452545,0.516259,0.509521,2.161885,0.658630,0.097962,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,42,3013109,74.817238,7c6028da2f1b1977953d54146683f4a2020459c88d115c...
1,0.518056,0.458766,0.455529,0.514140,0.477235,0.458766,0.527815,0.518056,2.077294,0.640507,0.095295,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,2024,3013109,79.260899,e13f8002041a02ed9aef21a8c4818e3c1ecde66d6afb47...
2,0.528562,0.464204,0.456500,0.525625,0.473112,0.464204,0.540633,0.528562,2.107729,0.645030,0.096736,wavestarnet_big_transfer_v2,WavestarNet-Big-Transfer-V2,3407,3013109,79.017946,a9cc947a81384fc367471ec547de760a32cd07ac44a2f5...
3,0.394616,0.342907,0.330650,0.388835,0.341059,0.342907,0.399048,0.394616,2.807080,0.788530,0.113263,efficientnet_b0,EfficientNet-B0,42,4142053,62.114273,2b74bfc30367d17021542380f0b5b32b4e0b3cf9e5fa14...
4,0.424819,0.370277,0.357334,0.415650,0.362887,0.370277,0.422336,0.424819,2.675055,0.749258,0.084823,efficientnet_b0,EfficientNet-B0,2024,4142053,60.932860,2dc58eb828a298d80dfb861ed4715cfce23133ec507732...
5,0.416284,0.355132,0.345620,0.407621,0.356357,0.355132,0.414282,0.416284,2.717120,0.755982,0.087904,efficientnet_b0,EfficientNet-B0,3407,4142053,60.708567,4fd703cc239f2cbf0ebfdf4aa01230d65b5515ee31970a...
6,0.445174,0.370832,0.363356,0.436778,0.379119,0.370832,0.443181,0.445174,2.553959,0.716655,0.075336,efficientnet_b3,EfficientNet-B3,42,10857617,102.337409,da5bdcc345339fdd914f2d52575e591cce7b70484cad51...
7,0.448457,0.390987,0.375624,0.439808,0.378339,0.390987,0.446310,0.448457,2.469685,0.710648,0.059880,efficientnet_b3,EfficientNet-B3,2024,10857617,105.039986,68c9480bd3a314c5276d087e6103e891659c194367e65b...
8,0.451740,0.382237,0.370048,0.442496,0.372686,0.382237,0.447051,0.451740,2.497885,0.711038,0.067599,efficientnet_b3,EfficientNet-B3,3407,10857617,108.404795,5d0dd6b454fd532edb23fbfa85f4a141ca0eae5b63f094...
9,0.395272,0.344148,0.333939,0.389722,0.347613,0.344148,0.402005,0.395272,2.785718,0.778366,0.128421,mobilenetv3_large_100,MobileNetV3-Large,42,4336537,48.228077,d62fe7a14ccaee3fce1990459ed1298d0cfdf6b556d6be...


**Saved to:** `/run/media/sha-weed/adc6c87e-0116-4a25-a6e9-8555193c1e51/Research/Agrivision4/plantwild/results/sealed_test/plantwild_scratch_final_holdout_v1`

If final evaluation completes but the notebook is not saved, set `RUN_MODE = 'report_existing'` and rerun all cells. That mode checks every published artifact hash and reloads the tables without reading images or running inference.